In [ ]:
import sqlite3
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path("..") / "data"
if not DATA_DIR.exists():
    DATA_DIR = Path("data")

SUBMISSION_DIR = DATA_DIR.parent / "submission"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

# Carga de la tabla drivers

drivers = pd.read_csv(
    DATA_DIR / "drivers.csv",
    sep=",",
    thousands=None,
    decimal=".",
)
drivers.head()


In [ ]:
# Carga de la tabla timesheet

timesheet = pd.read_csv(
    DATA_DIR / "timesheet.csv",
    sep=",",
    thousands=None,
    decimal=".",
)
timesheet.head()


In [ ]:
# Carga las tablas en una base SQLite en memoria

connection = sqlite3.connect(":memory:")
drivers.to_sql("drivers", connection, index=False, if_exists="replace")
timesheet.to_sql("timesheet", connection, index=False, if_exists="replace")


In [ ]:
# Totales de horas y millas por conductor, con su nombre

summary = pd.read_sql(
    """
    SELECT
        t.driverId AS driver_id,
        SUM(t."hours-logged") AS hours_logged,
        SUM(t."miles-logged") AS miles_logged,
        d.name AS name
    FROM timesheet AS t
    JOIN drivers AS d ON d.driverId = t.driverId
    GROUP BY t.driverId, d.name
    ORDER BY t.driverId
    """,
    connection,
)
summary


In [ ]:
summary.to_csv(SUBMISSION_DIR / "summary.csv", index=False)


In [ ]:
# Top 10 conductores por horas registradas

top10 = summary.nlargest(10, "hours_logged")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top10["name"], top10["hours_logged"], color="#3b82f6")
ax.invert_yaxis()
ax.set_xlabel("Horas registradas")
ax.set_title("Top 10 conductores por horas registradas")
fig.tight_layout()
fig.savefig(SUBMISSION_DIR / "top10_drivers.png", dpi=150)
plt.show()


In [ ]:
connection.close()
